In [15]:
# gernate png stegos
import os
import numpy as np
from PIL import Image
import conseal as cl
from conseal.lsb._costmap import Change


def gen(cover_dir, payload_rate, max_files):
    print(
        "running gen with dir",
        cover_dir,
        "payload: ",
        payload_rate,
        "max_files:",
        max_files,
    )
    payload_percentage = int(payload_rate * 100)

    methods = {
        "hill": cl.hill.simulate_single_channel,
        "hugo": cl.hugo.simulate_single_channel,
        "lsb": cl.lsb.simulate,
        "suniward": cl.suniward.simulate_single_channel,
        "wow": cl.wow.simulate_single_channel,
    }

    for name, _ in methods.items():
        if name != "lsb":
            output_base = f"data/{name}/{payload_percentage:02d}"
            os.makedirs(output_base, exist_ok=True)
        else:
            os.makedirs(f"data/lsbr/{payload_percentage:02d}", exist_ok=True)
            os.makedirs(f"data/lsbm/{payload_percentage:02d}", exist_ok=True)

    i = 0
    for filename in os.listdir(cover_dir)[:max_files]:
        if i % 5 == 0:
            print("Generated: ", i, "/", max_files)
        i += 1
        img_path = os.path.join(cover_dir, filename)
        cover = np.array(Image.open(img_path).convert("L"))
        N = cover.size
        M = int(N * payload_rate)

        for name, embedding_method in methods.items():

            if name == "lsb":
                stego_lsbr = embedding_method(
                    cover, modify=Change.LSB_REPLACEMENT, alpha=payload_rate
                )
                stego_lsbm = embedding_method(
                    cover, modify=Change.LSB_MATCHING, alpha=payload_rate
                )

                save_path_lsbm = os.path.join(
                    f"data/lsbm/{payload_percentage:02d}", filename
                )
                save_path_lsbr = os.path.join(
                    f"data/lsbr/{payload_percentage:02d}", filename
                )

                if not os.path.exists(save_path_lsbm):
                    Image.fromarray(stego_lsbm).save(save_path_lsbm)
                else:
                    print(f"LSBM bereits vorhanden: {save_path_lsbm}")

                if not os.path.exists(save_path_lsbr):
                    Image.fromarray(stego_lsbr).save(save_path_lsbr)
                else:
                    print(f"LSBR bereits vorhanden: {save_path_lsbr}")
            else:
                output_base = f"data/{name}/{payload_percentage:02d}"
                save_path = os.path.join(output_base, filename)

                if not os.path.exists(save_path):
                    stego = embedding_method(cover, alpha=payload_rate)
                    Image.fromarray(stego).save(save_path)
                else:
                    print(f"{name} bereits vorhanden: {save_path}")

    print(f"Benchmark-Datensatz erfolgreich erstellt in: {output_base}")


# Setup
cover_dir = "../data/boss_cover"
payload_rate = 0.45
max_files = 100
gen(cover_dir, payload_rate, max_files)

gen:  1.png
gen:  10.png
gen:  100.png
gen:  1000.png
gen:  1001.png
gen:  1002.png
gen:  1003.png
gen:  1004.png
gen:  1005.png
gen:  1006.png
gen:  1007.png
gen:  1008.png
gen:  1009.png
gen:  101.png
gen:  1010.png
gen:  1011.png
gen:  1012.png
gen:  1013.png
gen:  1014.png
gen:  1015.png
gen:  1016.png
gen:  1017.png
gen:  1018.png
gen:  1019.png
gen:  102.png
gen:  1020.png
gen:  1021.png
gen:  1022.png
gen:  1023.png
gen:  1024.png
gen:  1025.png
gen:  1026.png
gen:  1027.png
gen:  1028.png
gen:  1029.png
gen:  103.png
gen:  1030.png
gen:  1031.png
gen:  1032.png
gen:  1033.png
gen:  1034.png
gen:  1035.png
gen:  1036.png
gen:  1037.png
gen:  1038.png
gen:  1039.png
gen:  104.png
gen:  1040.png
gen:  1041.png
gen:  1042.png
gen:  1043.png
gen:  1044.png
gen:  1045.png
gen:  1046.png
gen:  1047.png
gen:  1048.png
gen:  1049.png
gen:  105.png
gen:  1050.png
gen:  1051.png
gen:  1052.png
gen:  1053.png
gen:  1054.png
gen:  1055.png
gen:  1056.png
gen:  1057.png
gen:  1058.png
gen:  

In [2]:
python -c "from conseal.lsb import Change; print(Change.__members__)"

In [17]:
# generate jpeg stegos
import os
import numpy as np
import jpeglib
import conseal as cl
from conseal.lsb._costmap import Change
from PIL import Image


def convert_png_to_jpeg(src_path, dst_path, quality=75):
    img = Image.open(src_path).convert("L")
    img.save(dst_path, "JPEG", quality=quality, subsampling=0)


def gen(cover_dir, jpeg_output_dir, payload_rate, max_files):
    print(
        "running gen with dir",
        cover_dir,
        "payload:",
        payload_rate,
        "max_files:",
        max_files,
    )
    payload_percentage = int(payload_rate * 100)

    # Jede Methode bekommt (dct, spatial, qt) — ungenutzte Parameter werden ignoriert
    methods = {
        "juniward": lambda dct, spatial, qt: cl.juniward.simulate_single_channel(
            x0=spatial, y0=dct, qt=qt, alpha=payload_rate
        ),
        "uerd": lambda dct, spatial, qt: cl.uerd.simulate_single_channel(
            y0=dct, qt=qt, alpha=payload_rate
        ),
        "ebs": lambda dct, spatial, qt: cl.ebs.simulate_single_channel(
            y0=dct, qt=qt, alpha=payload_rate
        ),
        "nsf5": lambda dct, spatial, qt: cl.nsF5.simulate_single_channel(
            y0=dct, alpha=payload_rate
        ),
        "f5": lambda dct, spatial, qt: cl.F5.simulate_single_channel(
            y0=dct, alpha=payload_rate
        ),
        "lsb": lambda dct, spatial, qt: cl.lsb.simulate(
            dct, modify=Change.LSB_REPLACEMENT, alpha=payload_rate
        ),
    }

    for method_name in methods:
        os.makedirs(
            f"data/jpeg/{method_name}/{payload_percentage:02d}", exist_ok=True
        )
    os.makedirs(jpeg_output_dir, exist_ok=True)

    # PNG → JPEG
    png_files = [
        f for f in os.listdir(cover_dir) if f.lower().endswith(".png")
    ][:max_files]
    for filename in png_files:
        jpg_name = os.path.splitext(filename)[0] + ".jpg"
        dst_path = os.path.join(jpeg_output_dir, jpg_name)
        if not os.path.exists(dst_path):
            convert_png_to_jpeg(
                os.path.join(cover_dir, filename), dst_path, quality=75
            )

    cover_files = [
        f for f in os.listdir(jpeg_output_dir) if f.lower().endswith(".jpg")
    ][:max_files]

    i = 0
    for filename in cover_files:
        if i % 5 == 0:
            print("Generated:", i, "/", max_files)
        i += 1

        cover_path = os.path.join(jpeg_output_dir, filename)

        jpeg = jpeglib.read_dct(cover_path)
        cover_dct = jpeg.Y
        qtable = jpeg.qt[0]
        jpeg_spatial = jpeglib.read_spatial(cover_path)
        cover_spatial = jpeg_spatial.spatial[:, :, 0]

        for method_name, embed_fn in methods.items():
            save_path = os.path.join(
                f"data/jpeg/{method_name}/{payload_percentage:02d}", filename
            )
            if os.path.exists(save_path):
                print(f"{method_name} bereits vorhanden: {save_path}")
                continue

            stego_dct = embed_fn(cover_dct, cover_spatial, qtable)
            jpeg_out = jpeglib.read_dct(cover_path)
            jpeg_out.Y = stego_dct
            jpeg_out.write_dct(save_path)

    print(
        f"Benchmark-Datensatz erfolgreich erstellt in: data/jpeg/*/{payload_percentage:02d}/"
    )


# Setup
for payload_rate in [
    0.1,
    0.15,
    0.2,
    0.25,
    0.3,
    0.35,
    0.4,
    0.45,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1,
]:
    cover_dir = "../data/boss_cover"
    jpeg_output_dir = "./data/jpeg/cover"
    max_files = 100
    gen(cover_dir, jpeg_output_dir, payload_rate, max_files)

running gen with dir ../data/boss_cover payload: 0.1 max_files: 100
Generated: 0 / 100
Generated: 5 / 100
Generated: 10 / 100
Generated: 15 / 100
Generated: 20 / 100


/home/feierabe/venv_info_projekt/lib/python3.12/site-packages/conseal/simulate/_optim.py:268: RuntimeWarning: optimization might not have converged
  warnings.warn("optimization might not have converged", RuntimeWarning)


Generated: 25 / 100
Generated: 30 / 100
Generated: 35 / 100
Generated: 40 / 100
Generated: 45 / 100
Generated: 50 / 100
Generated: 55 / 100
Generated: 60 / 100
Generated: 65 / 100
Generated: 70 / 100
Generated: 75 / 100
Generated: 80 / 100
Generated: 85 / 100
Generated: 90 / 100
Generated: 95 / 100
Benchmark-Datensatz erfolgreich erstellt in: data/jpeg/*/10/
running gen with dir ../data/boss_cover payload: 0.15 max_files: 100
Generated: 0 / 100
Generated: 5 / 100
Generated: 10 / 100
Generated: 15 / 100
Generated: 20 / 100
Generated: 25 / 100
Generated: 30 / 100
Generated: 35 / 100
Generated: 40 / 100
Generated: 45 / 100
Generated: 50 / 100
Generated: 55 / 100
Generated: 60 / 100
Generated: 65 / 100
Generated: 70 / 100
Generated: 75 / 100
Generated: 80 / 100
Generated: 85 / 100
Generated: 90 / 100
Generated: 95 / 100
Benchmark-Datensatz erfolgreich erstellt in: data/jpeg/*/15/
running gen with dir ../data/boss_cover payload: 0.2 max_files: 100
Generated: 0 / 100
Generated: 5 / 100
Gener